In [1]:
import os
import re
import gradio as gr
from pathlib import Path

def clean_filename(filename):
    """
    Clean filename by replacing German umlauts and special characters.
    
    Args:
        filename: Original filename
    
    Returns:
        Cleaned filename
    """
    # Separate name and extension
    name, ext = os.path.splitext(filename)
    
    # Define German umlaut replacements
    umlaut_map = {
        'ä': 'ae', 'Ä': 'Ae',
        'ö': 'oe', 'Ö': 'Oe',
        'ü': 'ue', 'Ü': 'Ue',
        'ß': 'ss'
    }
    
    # Replace German umlauts
    for umlaut, replacement in umlaut_map.items():
        name = name.replace(umlaut, replacement)
    
    # Replace spaces with hyphens
    name = name.replace(' ', '-')
    
    # Replace multiple consecutive hyphens with a single hyphen
    name = re.sub(r'-+', '-', name)
    
    # Remove any special characters except letters, numbers, hyphens, and underscores
    # Replace them with hyphens
    name = re.sub(r'[^a-zA-Z0-9_-]', '-', name)
    
    # Clean up multiple consecutive hyphens again
    name = re.sub(r'-+', '-', name)
    
    # Remove leading/trailing hyphens
    name = name.strip('-')
    
    # Return cleaned filename with extension
    return name + ext

def rename_files(folder_path):
    """
    Rename all PDF and image files in the specified folder.
    
    Args:
        folder_path: Path to the folder containing files
    
    Returns:
        Status message with details of renamed files
    """
    if not folder_path:
        return "Please select a folder.", []
    
    # Check if folder exists
    if not os.path.exists(folder_path):
        return f"Error: Folder '{folder_path}' does not exist.", []
    
    if not os.path.isdir(folder_path):
        return f"Error: '{folder_path}' is not a directory.", []
    
    # Define supported file extensions
    supported_extensions = {
        '.pdf',
        '.jpg', '.jpeg',
        '.png',
        '.gif',
        '.bmp',
        '.tiff', '.tif',
        '.webp',
        '.svg'
    }
    
    # Get all supported files in the folder
    all_files = [f for f in os.listdir(folder_path) 
                 if os.path.splitext(f)[1].lower() in supported_extensions]
    
    if not all_files:
        return "No PDF or image files found in the selected folder.", []
    
    renamed_files = []
    errors = []
    skipped = []
    
    for old_name in all_files:
        new_name = clean_filename(old_name)
        
        # Check if renaming is needed
        if old_name == new_name:
            skipped.append(old_name)
            continue
        
        old_path = os.path.join(folder_path, old_name)
        new_path = os.path.join(folder_path, new_name)
        
        # Handle duplicate filenames
        if os.path.exists(new_path):
            # Add number suffix to make it unique
            base_name, ext = os.path.splitext(new_name)
            counter = 1
            while os.path.exists(os.path.join(folder_path, f"{base_name}-{counter}{ext}")):
                counter += 1
            new_name = f"{base_name}-{counter}{ext}"
            new_path = os.path.join(folder_path, new_name)
        
        try:
            os.rename(old_path, new_path)
            renamed_files.append([old_name, new_name])
        except Exception as e:
            errors.append(f"Error renaming '{old_name}': {str(e)}")
    
    # Create status message
    status_parts = []
    
    if renamed_files:
        status_parts.append(f"✅ Successfully renamed {len(renamed_files)} file(s)")
    
    if skipped:
        status_parts.append(f"⏩ Skipped {len(skipped)} file(s) (no changes needed)")
    
    if errors:
        status_parts.append(f"❌ {len(errors)} error(s) occurred")
        for error in errors:
            status_parts.append(f"  • {error}")
    
    if not renamed_files and not errors:
        if skipped:
            status_parts.append("All files already have clean names.")
        else:
            status_parts.append("No files were renamed.")
    
    status_message = "\n".join(status_parts)
    
    return status_message, renamed_files

def create_interface():
    """Create and return the Gradio interface."""
    
    with gr.Blocks(title="PDF and Image File Renamer") as interface:
        gr.Markdown(
            """
            # PDF and Image File Renamer
            
            This tool renames PDF and image files by:
            - Converting German umlauts (ä→ae, ö→oe, ü→ue, ß→ss)
            - Replacing spaces and special characters with hyphens (-)
            - Cleaning up multiple consecutive hyphens
            
            **Supported formats:** PDF, JPG, JPEG, PNG, GIF, BMP, TIFF, TIF, WEBP, SVG
            
            ### How to use:
            1. Enter the path to your folder containing files
            2. Click "Rename Files"
            3. Review the results in the table below
            """
        )
        
        with gr.Row():
            folder_input = gr.Textbox(
                label="Folder Path",
                placeholder="Enter the path to your folder (e.g., /Users/username/Documents/Images)",
                lines=1
            )
        
        with gr.Row():
            rename_button = gr.Button("Rename Files", variant="primary")
        
        with gr.Row():
            status_output = gr.Textbox(
                label="Status",
                lines=5,
                interactive=False
            )
        
        with gr.Row():
            results_table = gr.Dataframe(
                headers=["Original Name", "New Name"],
                label="Renamed Files",
                interactive=False
            )
        
        gr.Markdown(
            """
            ### Notes:
            - Only PDF and image files will be renamed
            - Files with clean names will be skipped
            - Duplicate names will get a number suffix (e.g., file-1.pdf, file-2.jpg)
            - Original files are renamed (no copies are created)
            """
        )
        
        # Connect the button to the function
        rename_button.click(
            fn=rename_files,
            inputs=folder_input,
            outputs=[status_output, results_table]
        )
    
    return interface

# Main execution
if __name__ == "__main__":
    # Create and launch the interface
    app = create_interface()
    app.launch(
        share=False,  # Set to True if you want to share the app publicly
    )

c:\Users\r.musawi\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.
